In [ ]:
import os
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
import cv2

os.environ["SM_FRAMEWORK"] = "tf.keras"
import segmentation_models as sm

In [46]:
metadata = pd.read_csv("data/metadata.csv")

metadata.head()

,Image,Mask
0,0.jpg,0.png
1,1.jpg,1.png
2,2.jpg,2.png
3,3.jpg,3.png
4,4.jpg,4.png


In [47]:
IMAGE_DIR = "data/Image"
MASK_DIR = "data/Mask"

image_paths = [
    os.path.join(IMAGE_DIR, img_name)
    for img_name in metadata["Image"]
]

mask_paths = [
    os.path.join(MASK_DIR, mask_name)
    for mask_name in metadata["Mask"]
]

In [48]:
from sklearn.model_selection import train_test_split

train_images, val_images, train_masks, val_masks = train_test_split(
    image_paths,
    mask_paths,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(train_images))
print("Validation samples:", len(val_images))

Training samples: 232
Validation samples: 58


In [49]:
print(train_images[:5])
print(train_masks[:5])

['data/Image/6.jpg', 'data/Image/1007.jpg', 'data/Image/1021.jpg', 'data/Image/1059.jpg', 'data/Image/2024.jpg']
['data/Mask/6.png', 'data/Mask/1007.png', 'data/Mask/1021.png', 'data/Mask/1059.png', 'data/Mask/2024.png']


In [50]:
IMG_SIZE = 256
BATCH_SIZE = 8

In [51]:
def load_single(img_path, mask_path):
    # cv2 load
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image.astype(np.float32) / 255.0

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = (mask > 127).astype(np.float32)
    mask = np.expand_dims(mask, axis=-1)

    return image, mask

# load everything into memory upfront — dataset is only 290 images, fits easily in 32GB RAM
all_images = []
all_masks  = []

for img_p, msk_p in zip(image_paths, mask_paths):
    img, msk = load_single(img_p, msk_p)
    all_images.append(img)
    all_masks.append(msk)

all_images = np.array(all_images)  # (290, 256, 256, 3)
all_masks  = np.array(all_masks)   # (290, 256, 256, 1)

print(f"Images: {all_images.shape}")
print(f"Masks:  {all_masks.shape}")

Images: (290, 256, 256, 3)
Masks:  (290, 256, 256, 1)


In [52]:
# split
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    all_images, all_masks, test_size=0.2, random_state=42
)

In [53]:
# tf datasets — no map, no numpy_function, no AutoGraph issues
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [54]:
# sanity check
for images, masks in train_dataset.take(1):
    print(f"Image batch: {images.shape}")  # (8, 256, 256, 3)
    print(f"Mask batch:  {masks.shape}")   # (8, 256, 256, 1)
    print(f"Mask unique: {np.unique(masks.numpy())}")  # [0. 1.]

Image batch: (8, 256, 256, 3)
Mask batch:  (8, 256, 256, 1)
Mask unique: [0. 1.]


In [55]:
for images, masks in train_dataset.take(1):
    print(images.shape)
    print(masks.shape)

(8, 256, 256, 3)
(8, 256, 256, 1)


In [56]:
model = sm.Unet(
    backbone_name="resnet34",
    encoder_weights="imagenet",
    classes=1,
    activation="sigmoid"
)

In [57]:
model.compile(
    optimizer='adam',
    loss=sm.losses.bce_dice_loss,
    metrics=[
        sm.metrics.iou_score,
        sm.metrics.f1_score
    ]
)

In [58]:
callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_f1-score",
        mode="max",
        save_best_only=True
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_f1-score",
        mode="max",
        patience=8,
        restore_best_weights=True
    )

]

In [59]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=callbacks
)

BATCH_SIZE = 8

Epoch 1/30
29/29 [==============================] - 90s 3s/step - loss: 0.7691 - iou_score: 0.5068 - f1-score: 0.6639 - val_loss: 8.0994 - val_iou_score: 0.0129 - val_f1-score: 0.0255
Epoch 2/30
29/29 [==============================] - 85s 3s/step - loss: 0.5798 - iou_score: 0.6244 - f1-score: 0.7677 - val_loss: 318520.6562 - val_iou_score: 0.4094 - val_f1-score: 0.5793
Epoch 3/30
29/29 [==============================] - 82s 3s/step - loss: 0.5342 - iou_score: 0.6503 - f1-score: 0.7870 - val_loss: 4.9393 - val_iou_score: 0.0369 - val_f1-score: 0.0712
Epoch 4/30
29/29 [==============================] - 82s 3s/step - loss: 0.4834 - iou_score: 0.6837 - f1-score: 0.8108 - val_loss: 1649.5793 - val_iou_score: 7.0181e-05 - val_f1-score: 1.4035e-04
Epoch 5/30
29/29 [==============================] - 82s 3s/step - loss: 0.4821 - iou_score: 0.6851 - f1-score: 0.8117 - val_loss: 2.8679 - val_iou_score: 0.4079 - val_f1-score: 0.5770
Epoch 6/30
29/29 [==============================] - 86s 3s/step 